# 1. Download Final Gaze Sweeps
This cell connects to the W&B project `PCS_ET_v22` and fetches the results for sweeps:
* `tpufsm5n`, `uqltfg5w` (DINOv3 and DeiT III)
* `sdoclfmj`, `b4c9o2cn` (CLIP)

It extracts hyperparameters, validation accuracy, test accuracy, and the `run_id`. 
The results are saved in `results/gaze_sweeps_data.csv`.

In [1]:
import wandb
import pandas as pd
import numpy as np
import os

# 1. Setup API
api = wandb.Api()
entity = "luis-perdigao-instituto-superior-t-cnico"
project = "PCS_ET_v22"
target_sweeps = ["tpufsm5n", "uqltfg5w", "sdoclfmj", "b4c9o2cn"]

print(f"Connecting to {entity}/{project}...")

data_list = []

for sweep_id in target_sweeps:
    print(f"Fetching runs from sweep: {sweep_id}...")
    try:
        sweep = api.sweep(f"{entity}/{project}/{sweep_id}")
        runs = sweep.runs
    except Exception as e:
        print(f"Error accessing sweep {sweep_id}: {e}")
        continue

    print(f"  Found {len(runs)} runs. Scanning history...")

    for run in runs:
        config = {k: v for k, v in run.config.items() if not k.startswith('_')}
        
        # ---------------------------------------------------------
        # NEW FILTERING LOGIC: Ignore unwanted backbones per sweep
        # ---------------------------------------------------------
        bb = str(config.get("backbone", "unknown")).lower()
        
        # If it's the DINO/DeiT sweeps, ignore CLIP/ViT-Base runs
        if sweep_id in ["tpufsm5n", "uqltfg5w"]:
            if "clip" in bb or ("vit_base" in bb and "dinov3" not in bb):
                continue 
                
        # If it's the CLIP sweeps, ignore DINO/DeiT runs (just to be completely safe)
        if sweep_id in ["sdoclfmj", "b4c9o2cn"]:
            if "dinov3" in bb or "deit" in bb:
                continue 
        # ---------------------------------------------------------

        # Fetch History
        keys = ["accuracy_validation", "accuracy_test", "c_accuracy_test"]
        history = pd.DataFrame(run.scan_history(keys=keys))
        
        val_rank_acc = np.nan
        test_rank_acc = np.nan
        test_class_acc = np.nan
        
        if not history.empty and "accuracy_validation" in history.columns:
            best_epoch_idx = history["accuracy_validation"].idxmax()
            val_rank_acc = history["accuracy_validation"].max()
            
            if "accuracy_test" in history.columns:
                test_rank_acc = history.loc[best_epoch_idx, "accuracy_test"]
            if "c_accuracy_test" in history.columns:
                test_class_acc = history.loc[best_epoch_idx, "c_accuracy_test"]
        else:
            # Fallback
            val_rank_acc = run.summary.get("max_accuracy_validation", run.summary.get("accuracy_validation", np.nan))
            test_rank_acc = run.summary.get("max_accuracy_test", run.summary.get("accuracy_test", np.nan))
            test_class_acc = run.summary.get("c_accuracy_test", np.nan)

        data_list.append({
            "run_id": run.id,
            "run_name": run.name,
            "sweep_id": sweep_id,
            "backbone": config.get("backbone", "unknown"),
            "gaze_mode": config.get("gaze_mode", "off"),
            "attention_mode": config.get("attention_mode", "raw"),
            "seed": config.get("seed", -1),
            "val_rank_acc": val_rank_acc,
            "test_rank_acc": test_rank_acc,
            "test_class_acc": test_class_acc
        })

# 2. Ensure directory exists and save to CSV
os.makedirs("results", exist_ok=True)
filename = "results/gaze_sweeps_data.csv"

df = pd.DataFrame(data_list)
df.to_csv(filename, index=False)
print(f"Done! Saved {len(df)} correctly filtered runs to '{filename}'.")

wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: luis-perdigao (luis-perdigao-instituto-superior-t-cnico) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Connecting to luis-perdigao-instituto-superior-t-cnico/PCS_ET_v22...
Fetching runs from sweep: tpufsm5n...
  Found 90 runs. Scanning history...
Fetching runs from sweep: uqltfg5w...
  Found 60 runs. Scanning history...
Fetching runs from sweep: sdoclfmj...
  Found 30 runs. Scanning history...
Fetching runs from sweep: b4c9o2cn...
  Found 20 runs. Scanning history...
Done! Saved 150 correctly filtered runs to 'results/gaze_sweeps_data.csv'.


# 2. Compute Averages and Generate Tables
This cell loads `results/gaze_sweeps_data.csv`, aggregates the runs by Method and Attention Mode, and computes the 95% Confidence Intervals.

It outputs:
1. **LaTeX Table**: Fully formatted with citations and confidence intervals.
2. **Notebook Table**: A styled Pandas DataFrame for quick viewing.

In [2]:
import pandas as pd
import numpy as np
from IPython.display import display

def map_method(row):
    g_mode = str(row['gaze_mode']).lower()
    if g_mode in ['off', 'disable']:
        return "Baseline"
    elif g_mode == 'guide':
        return "GII injection \\cite{Chen2026GIIViT}"
    elif g_mode == 'egvit':
        return "EGViT \\cite{Ma2023EGViT}"
    elif g_mode == 'align':
        return "EG-PCS-Net (Ours)"
    else:
        return g_mode.capitalize()

def map_attn(row):
    g_mode = str(row['gaze_mode']).lower()
    if g_mode == 'align':
        return str(row.get('attention_mode', 'raw')).capitalize()
    return "--"

def generate_gaze_tables(csv_path="results/gaze_sweeps_data.csv"):
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        return "Error: CSV file not found. Run Cell 1 first."

    # 1. Clean Names & Assign Method / Attention
    backbone_map = {
        'dinov3_vitb16': 'DINOv3',
        'deit3_base_patch16_224': 'DeiT III',
        'vit_base_patch16_224': 'CLIP',       
        'vit_base_patch16_clip_224': 'CLIP'  
    }
    df['backbone'] = df['backbone'].map(backbone_map).fillna(df['backbone'])
    df['Method'] = df.apply(map_method, axis=1)
    df['Attn Mode'] = df.apply(map_attn, axis=1)

    # 2. Aggregate
    df_agg = df.groupby(['backbone', 'Method', 'Attn Mode']).agg(
        rank_mean=('test_rank_acc', 'mean'),
        rank_std=('test_rank_acc', 'std'),
        rank_count=('test_rank_acc', 'count'),
        class_mean=('test_class_acc', 'mean'),
        class_std=('test_class_acc', 'std'),
        class_count=('test_class_acc', 'count')
    ).reset_index()

    # 95% CI
    df_agg['rank_ci'] = 1.96 * (df_agg['rank_std'] / np.sqrt(df_agg['rank_count']))
    df_agg['class_ci'] = 1.96 * (df_agg['class_std'] / np.sqrt(df_agg['class_count']))

    for col in ['rank_mean', 'rank_ci', 'class_mean', 'class_ci']:
        df_agg[col] *= 100

    # 3. Create a safe String Index
    df_agg['SafeKey'] = df_agg['Method'] + "@@@" + df_agg['Attn Mode']
    
    pivot_rank_mean = df_agg.pivot_table(index='SafeKey', columns='backbone', values='rank_mean')
    pivot_rank_ci = df_agg.pivot_table(index='SafeKey', columns='backbone', values='rank_ci')
    pivot_class_mean = df_agg.pivot_table(index='SafeKey', columns='backbone', values='class_mean')
    pivot_class_ci = df_agg.pivot_table(index='SafeKey', columns='backbone', values='class_ci')

    # Desired Row Order mappings
    row_order = [
        ("Baseline", "--"),
        ("GII injection \\cite{Chen2026GIIViT}", "--"),
        ("EGViT \\cite{Ma2023EGViT}", "--"),
        ("EG-PCS-Net (Ours)", "Raw"),
        ("EG-PCS-Net (Ours)", "Rollout")
    ]
    
    existing_keys = []
    display_tuples = []
    for method, attn in row_order:
        k = f"{method}@@@{attn}"
        if k in pivot_rank_mean.index:
            existing_keys.append(k)
            display_tuples.append((method, attn))
            
    pivot_rank_mean = pivot_rank_mean.reindex(existing_keys)
    pivot_rank_ci = pivot_rank_ci.reindex(existing_keys)
    pivot_class_mean = pivot_class_mean.reindex(existing_keys)
    pivot_class_ci = pivot_class_ci.reindex(existing_keys)

    cols = ['DINOv3', 'DeiT III', 'CLIP'] 
    cols = [c for c in cols if c in pivot_rank_mean.columns]

    # PART A: LaTeX Table
    latex_str = []
    latex_str.append("\\begin{table*}[ht]")
    latex_str.append("\\centering")
    latex_str.append("\\caption{Average Accuracy (Rank / Class) with 95\\% confidence intervals over tested seeds.}")
    latex_str.append("\\label{tab:gaze_modes_final}")
    
    latex_str.append("\\begin{tabular}{llccc}")
    latex_str.append("\\toprule")
    latex_str.append(f"Method & Attention Mode & {' & '.join(cols)} \\\\")
    latex_str.append("\\midrule")
    
    for safe_key in existing_keys:
        method, attn = safe_key.split("@@@")
        row_values = []
        for col in cols:
            val_r_mean = pivot_rank_mean.loc[safe_key, col]
            val_r_ci = pivot_rank_ci.loc[safe_key, col]
            val_c_mean = pivot_class_mean.loc[safe_key, col]
            val_c_ci = pivot_class_ci.loc[safe_key, col]
            
            if np.isnan(val_r_mean) or np.isnan(val_c_mean):
                row_values.append("-")
            else:
                s_r_mean_str = f"{val_r_mean:.2f}"
                s_c_mean_str = f"{val_c_mean:.2f}"
                
                col_max_r = pivot_rank_mean[col].max()
                col_max_c = pivot_class_mean[col].max()
                
                if abs(val_r_mean - col_max_r) < 1e-9: s_r_mean_str = f"\\textbf{{{s_r_mean_str}}}"
                if abs(val_c_mean - col_max_c) < 1e-9: s_c_mean_str = f"\\textbf{{{s_c_mean_str}}}"
                
                final_str = f"{s_r_mean_str}$_{{{{\\pm{val_r_ci:.2f}}}}}$/{s_c_mean_str}$_{{{{\\pm{val_c_ci:.2f}}}}}$"
                row_values.append(final_str)
        
        latex_str.append(f"{method} & {attn} & {' & '.join(row_values)} \\\\")

    latex_str.append("\\bottomrule")
    latex_str.append("\\end{tabular}")
    latex_str.append("\\end{table*}")

    print("% " + "="*50)
    print("% LATEX TABLE (Copy to Overleaf)")
    print("% " + "="*50)
    print("\n".join(latex_str))
    print("\n")

    # PART B: Pandas Display
    idx = pd.MultiIndex.from_tuples(display_tuples, names=["Method", "Attention Mode"])
    display_df = pd.DataFrame(index=idx, columns=cols)
    
    for safe_key, (method, attn) in zip(existing_keys, display_tuples):
        for col in cols:
            val_r_mean = pivot_rank_mean.loc[safe_key, col]
            val_r_ci = pivot_rank_ci.loc[safe_key, col]
            val_c_mean = pivot_class_mean.loc[safe_key, col]
            val_c_ci = pivot_class_ci.loc[safe_key, col]
            
            if np.isnan(val_r_mean) or np.isnan(val_c_mean):
                display_df.loc[(method, attn), col] = "-"
            else:
                display_df.loc[(method, attn), col] = f"{val_r_mean:.2f}±{val_r_ci:.2f}% / {val_c_mean:.2f}±{val_c_ci:.2f}%"

    print("# " + "="*50)
    print("# NOTEBOOK DISPLAY (Rank / Class)")
    print("# " + "="*50)
    display(display_df)

# Run
generate_gaze_tables()

% ==================================================
% LATEX TABLE (Copy to Overleaf)
% ==================================================
\begin{table*}[ht]
\centering
\caption{Average Accuracy (Rank / Class) with 95\% confidence intervals over tested seeds.}
\label{tab:gaze_modes_final}
\begin{tabular}{llccc}
\toprule
Method & Attention Mode & DINOv3 & DeiT III & CLIP \\
\midrule
Baseline & -- & 74.73$_{{\pm0.59}}$/74.06$_{{\pm0.84}}$ & 73.84$_{{\pm0.84}}$/73.20$_{{\pm1.14}}$ & 73.34$_{{\pm0.56}}$/72.11$_{{\pm1.78}}$ \\
GII injection \cite{Chen2026GIIViT} & -- & \textbf{74.88}$_{{\pm0.83}}$/74.55$_{{\pm0.75}}$ & 73.29$_{{\pm0.97}}$/72.82$_{{\pm1.18}}$ & \textbf{73.36}$_{{\pm0.74}}$/\textbf{73.42}$_{{\pm0.76}}$ \\
EGViT \cite{Ma2023EGViT} & -- & 74.41$_{{\pm0.57}}$/74.21$_{{\pm0.70}}$ & 73.94$_{{\pm0.69}}$/73.35$_{{\pm0.78}}$ & 73.03$_{{\pm0.96}}$/72.15$_{{\pm1.42}}$ \\
EG-PCS-Net (Ours) & Raw & 74.66$_{{\pm0.60}}$/\textbf{74.64}$_{{\pm0.52}}$ & 73.78$_{{\pm0.72}}$/73.52$_{{\pm0.84}}$

DINOv3  \
Method                              Attention Mode                              
Baseline                            --              74.73±0.59% / 74.06±0.84%   
GII injection \cite{Chen2026GIIViT} --              74.88±0.83% / 74.55±0.75%   
EGViT \cite{Ma2023EGViT}            --              74.41±0.57% / 74.21±0.70%   
EG-PCS-Net (Ours)                   Raw             74.66±0.60% / 74.64±0.52%   
                                    Rollout         74.32±0.51% / 74.60±0.56%   

                                                                     DeiT III  \
Method                              Attention Mode                              
Baseline                            --              73.84±0.84% / 73.20±1.14%   
GII injection \cite{Chen2026GIIViT} --              73.29±0.97% / 72.82±1.18%   
EGViT \cite{Ma2023EGViT}            --              73.94±0.69% / 73.35±0.78%   
EG-PCS-Net (Ours)                   Raw             73.78±0.72% / 73.52±0.84%   
                                    Rollout         74.20±0.88% / 73.93±0.93%   

                                                                         CLIP  
Method                              Attention Mode                             
Baseline                            --              73.34±0.56% / 72.11±1.78%  
GII injection \cite{Chen2026GIIViT} --              73.36±0.74% / 73.42±0.76%  
EGViT \cite{Ma2023EGViT}            --              73.03±0.96% / 72.15±1.42%  
EG-PCS-Net (Ours)                   Raw             73.32±0.82% / 73.06±0.79%  
                                    Rollout         72.71±0.65% / 71.68±1.50%

# 3. Generate Checkpoints Configuration
This cell automatically finds the best single run (W&B ID) for each `backbone` + `method` combination.
You can easily toggle `BEST_BY = "test"` or `"val"` to decide which metric dictates the "best" run.

It outputs the exact Python dictionary array format required for your evaluation scripts.

In [6]:
import pandas as pd

# ==========================================
# CONFIGURATION
# Choose what metric determines the "best" run
BEST_BY = "val"  # Options: "val" or "test"
# ==========================================

def get_simple_backbone(bb_str):
    bb_str = str(bb_str).lower()
    if 'dinov3' in bb_str: return 'dinov3'
    if 'deit' in bb_str: return 'deit'
    if 'vit' in bb_str or 'clip' in bb_str: return 'clip'
    return 'unknown'

def get_simple_method(gaze_str):
    gaze_str = str(gaze_str).lower()
    if gaze_str in ['off', 'disable']: return 'baseline'
    if gaze_str == 'guide': return 'guide'
    if gaze_str == 'egvit': return 'egvit'
    if gaze_str == 'align': return 'align'
    return 'unknown'

def get_simple_attn_mode(mode_str):
    mode_str = str(mode_str).lower()
    if 'rollout' in mode_str: return 'rollout'
    if 'raw' in mode_str: return 'raw'
    return 'unknown'

def generate_checkpoints_dict(csv_path="results/gaze_sweeps_data.csv", best_by="val"):
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        print("Error: CSV file not found. Run Cell 1 first.")
        return

    # Map to simple tags
    df['simple_bb'] = df['backbone'].apply(get_simple_backbone)
    df['simple_meth'] = df['gaze_mode'].apply(get_simple_method)
    
    # Check what the attention mode column is called in your CSV
    attn_col = 'attn_mode' if 'attn_mode' in df.columns else 'attention_mode' if 'attention_mode' in df.columns else None
    if attn_col:
        df['simple_attn'] = df[attn_col].apply(get_simple_attn_mode)
    else:
        # Fallback if the column isn't found
        df['simple_attn'] = 'unknown' 
        print("Warning: Attention mode column not found in CSV. Expected 'attn_mode' or 'attention_mode'.")

    # Define combinations in the exact order requested: (backbone, method, attn_mode)
    combinations = [
        ("dinov3", "baseline", None), ("dinov3", "guide", None), ("dinov3", "egvit", None), 
        ("dinov3", "align", "raw"), ("dinov3", "align", "rollout"),
        
        ("clip", "baseline", None), ("clip", "guide", None), ("clip", "egvit", None), 
        ("clip", "align", "raw"), ("clip", "align", "rollout"),
        
        ("deit", "baseline", None), ("deit", "guide", None), ("deit", "egvit", None), 
        ("deit", "align", "raw"), ("deit", "align", "rollout")
    ]
    
    metric_col = "val_rank_acc" if best_by == "val" else "test_rank_acc"
    
    print("CHECKPOINTS = [")
    for bb, meth, attn in combinations:
        # Filter based on whether we care about the attention mode
        if attn is not None:
            subset = df[(df['simple_bb'] == bb) & (df['simple_meth'] == meth) & (df['simple_attn'] == attn)]
            tag = f"{bb}_{meth}_{attn}"
        else:
            subset = df[(df['simple_bb'] == bb) & (df['simple_meth'] == meth)]
            tag = f"{bb}_{meth}"
        
        if subset.empty:
            best_id = "NOT_FOUND"
            val_score = 0.0
            test_score = 0.0
        else:
            # Find row with max score
            best_idx = subset[metric_col].idxmax()
            best_row = subset.loc[best_idx]
            
            best_id = best_row['run_id']
            val_score = best_row['val_rank_acc'] * 100
            test_score = best_row['test_rank_acc'] * 100
        
        print(f"    {{")
        print(f"        \"tag\": \"{tag}\",")
        print(f"        \"wandb_run_id\": \"{best_id}\", # val: {val_score:.2f}%, test: {test_score:.2f}%")
        print(f"        \"checkpoint\": None,")
        print(f"        \"checkpoint_kind\": \"best\",")
        print(f"    }},")
    print("]")

# Run
generate_checkpoints_dict(best_by=BEST_BY)

CHECKPOINTS = [
    {
        "tag": "dinov3_baseline",
        "wandb_run_id": "4evy0w94", # val: 77.95%, test: 74.49%
        "checkpoint": None,
        "checkpoint_kind": "best",
    },
    {
        "tag": "dinov3_guide",
        "wandb_run_id": "79fedrh2", # val: 77.78%, test: 74.06%
        "checkpoint": None,
        "checkpoint_kind": "best",
    },
    {
        "tag": "dinov3_egvit",
        "wandb_run_id": "k4v5o50x", # val: 76.74%, test: 73.72%
        "checkpoint": None,
        "checkpoint_kind": "best",
    },
    {
        "tag": "dinov3_align_raw",
        "wandb_run_id": "fpz1g1lb", # val: 78.82%, test: 73.63%
        "checkpoint": None,
        "checkpoint_kind": "best",
    },
    {
        "tag": "dinov3_align_rollout",
        "wandb_run_id": "f7kn2l9r", # val: 77.95%, test: 74.40%
        "checkpoint": None,
        "checkpoint_kind": "best",
    },
    {
        "tag": "clip_baseline",
        "wandb_run_id": "xu652h29", # val: 76.56%, test: 73.97%
        "che